In [1]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs/detect/koi-detect/exp_real_100/weights/best.pt")
video_path = r"../data/real_test_data/video.mp4"

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'ขนาด: {w}x{h}, FPS: {fps}')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        frame,
        conf=0.3,
        verbose=False,
        device=0,
        imgsz=max(w, h)  # ใช้ขนาดจริงของวิดีโอเลย
    )

    for i, box in enumerate(results[0].boxes.xyxy):
        x1, y1, x2, y2 = map(int, box)
        conf = float(results[0].boxes.conf[i])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
        cv2.putText(frame, f"fish {conf:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    count = len(results[0].boxes)
    cv2.putText(frame, f"Total: {count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Koi Detection", frame)

    # เช็คว่ามีการกดปุ่ม Esc (27) หรือไม่
    key = cv2.waitKey(1) & 0xFF
    if key == 27: 
        print("Closing program...")
        break

cap.release()
cv2.destroyAllWindows()

ขนาด: 1920x1080, FPS: 59.56727421991237
Closing program...


In [6]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs/detect/koi-detect/exp_v2/weights/best.pt")
video_path = r"../data/real_test_data/video.mp4"

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'ขนาด: {w}x{h}, FPS: {fps}')

PROCESS_EVERY_N = 4  # ประมวลผลทุก 3 เฟรม (ปรับได้)
frame_count = 0
last_results = None  # เก็บผลลัพธ์ล่าสุดไว้แสดงในเฟรมที่ skip

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # ประมวลผลเฉพาะเฟรมที่หารลงตัว
    if frame_count % PROCESS_EVERY_N == 0:
        last_results = model.predict(
            frame,
            conf=0.3,
            verbose=False,
            device=0,
            half=True,        # FP16 ช่วย speed ขึ้นอีก
            imgsz=max(w, h)
        )

    # วาด bounding box จาก last_results (ถ้ามี)
    if last_results is not None:
        for i, box in enumerate(last_results[0].boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            conf = float(last_results[0].boxes.conf[i])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
            cv2.putText(frame, f"fish {conf:.2f}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        count = len(last_results[0].boxes)
        cv2.putText(frame, f"Total: {count}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Koi Detection", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        print("Closing program...")
        break

cap.release()
cv2.destroyAllWindows()

ขนาด: 1920x1080, FPS: 59.56727421991237
Closing program...


In [ ]:
import cv2
import numpy as np
import os
import glob
from ultralytics import YOLO

# --- การตั้งค่า (Settings) ---
MODEL_PATH = r"runs/detect/koi-detect/exp_100/weights/best.pt"
INPUT_DIR = r"../data/real_test_data"  # โฟลเดอร์หลักที่เก็บวิดีโอ
IGNORE_KEYWORDS = ["output", "result", "temp", "_processed"] # คำในชื่อไฟล์หรือ Folder ที่จะข้าม
PROCESS_EVERY_N = 4  # ประมวลผลทุกๆ 4 เฟรมเพื่อความลื่นไหล
GAMMA_VALUE = 0.7    # ค่า Gamma: น้อยกว่า 1.0 = มืดลง (ช่วยลดแสงสะท้อน), มากกว่า 1.0 = สว่างขึ้น

# --- ฟังก์ชันช่วย (Helper Functions) ---
def adjust_gamma(image, gamma=1.0):
    """ปรับค่า Gamma ของภาพเพื่อจัดการความสว่างและ Contrast"""
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255
                      for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

# 1. โหลด Model
print("Loading model...")
model = YOLO(MODEL_PATH)

# 2. ค้นหาไฟล์ .mp4 ทั้งหมดใน Folder และ Sub-folder
all_videos = glob.glob(os.path.join(INPUT_DIR, "**/*.mp4"), recursive=True)

# 3. กรองไฟล์ (Filter/Ignore)
valid_videos = []
for v in all_videos:
    # ถ้าใน Path มีคำที่สั่งให้ Ignore ให้ข้ามไป
    if any(key.lower() in v.lower() for key in IGNORE_KEYWORDS):
        continue
    valid_videos.append(v)

print(f"พบวิดีโอทั้งหมด: {len(valid_videos)} ไฟล์")

# 4. ลูปประมวลผลทีละไฟล์
for video_path in valid_videos:
    print(f"\nกำลังประมวลผล: {os.path.basename(video_path)}")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ไม่สามารถเปิดไฟล์ได้: {video_path}")
        continue

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    frame_count = 0
    last_results = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # --- ส่วน Image Processing (ลดแสง/ปรับ Contrast) ---
        # ปรับ Gamma ก่อนส่งให้ AI และแสดงผล
        processed_frame = adjust_gamma(frame, gamma=GAMMA_VALUE)

        # ประมวลผล AI เฉพาะเฟรมที่กำหนด
        if frame_count % PROCESS_EVERY_N == 0:
            last_results = model.predict(
                processed_frame,
                conf=0.3,
                verbose=False,
                device=0,   # ใช้ GPU (ถ้ามี)
                half=True,  # FP16 เพื่อความเร็ว
                imgsz=max(w, h)
            )

        # --- วาดผลลัพธ์ลงบนภาพที่ปรับแสงแล้ว ---
        if last_results is not None:
            for i, box in enumerate(last_results[0].boxes.xyxy):
                x1, y1, x2, y2 = map(int, box)
                conf = float(last_results[0].boxes.conf[i])
                
                # วาด Box และ Label
                cv2.rectangle(processed_frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
                cv2.putText(processed_frame, f"koi {conf:.2f}", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

            # แสดงจำนวนที่นับได้
            count = len(last_results[0].boxes)
            cv2.putText(processed_frame, f"Count: {count} | File: {os.path.basename(video_path)}", 
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # แสดงผลหน้าจอ
        cv2.imshow("Koi Detection System", processed_frame)

        # กด ESC เพื่อข้ามไฟล์นี้ หรือกด 'q' เพื่อปิดโปรแกรม
        key = cv2.waitKey(1) & 0xFF
        if key == 27: # ESC
            print("Skipping to next video...")
            break
        elif key == ord('q'):
            print("Closing program...")
            cap.release()
            cv2.destroyAllWindows()
            exit()

    cap.release()

cv2.destroyAllWindows()
print("Done!")

Loading model...
พบวิดีโอทั้งหมด: 1 ไฟล์

กำลังประมวลผล: video.mp4
Skipping to next video...
Done!
